# icetl quickstart

A DataFrame and SQL surface over Apache Iceberg, executed by DuckDB, on one machine.

**What to know before you start.** icetl follows Apache Spark 3.5 as a written
*specification* — `1/0` is NULL, a failed cast is NULL, `ORDER BY` puts nulls first
ascending. Nothing here runs on or requires Spark; it is a spec reference, and every
place the two engines disagree is written down in
[`compat/divergence.md`](../src/icetl/compat/divergence.md).

This notebook runs entirely offline against a local warehouse it builds itself, so it
needs no catalog and no MinIO. See [`01_read_real_table.ipynb`](01_read_real_table.ipynb)
for the same thing against a real REST catalog.

## 1. A warehouse to play in

`SqlCatalog` with sqlite metadata and a local-filesystem warehouse is a real Iceberg
catalog — the same one the test suite uses. Nothing in the rest of the notebook knows
it is not a REST catalog.

In [ ]:
import tempfile
from pathlib import Path

from pyiceberg.catalog.sql import SqlCatalog

from icetl.conf import CatalogSettings, EngineSettings, IcetlSettings
from icetl.sql import Session
from icetl.sql import functions as F

root = Path(tempfile.mkdtemp(prefix="icetl-quickstart-"))
catalog = SqlCatalog(
    "demo",
    uri=f"sqlite:///{(root / 'catalog.db').as_posix()}",
    warehouse=f"file://{root.as_posix()}",
)
catalog.create_namespace_if_not_exists("demo")

session = Session(
    settings=IcetlSettings(
        catalog=CatalogSettings(
            name="demo",
            type="sql",
            uri=f"sqlite:///{(root / 'catalog.db').as_posix()}",
            warehouse=f"file://{root.as_posix()}",
        ),
        engine=EngineSettings(temp_directory=str(root / "spill")),
        default_namespace=("demo",),
    ),
    catalog=catalog,
)
session

### Against your own catalog instead

`Session.builder` reads `.env` and the process environment, so the real thing needs no
code — see `.env.example` for every key:

```python
session = Session.builder.appName("quickstart").getOrCreate()
```

## 2. Getting data in

`createDataFrame` takes rows, dicts, pandas or Arrow. `write.saveAsTable` creates the
Iceberg table if it does not exist, and `partitionBy` gives the scan planner something
to prune with.

In [ ]:
rows = [
    (1, "AAPL", "2026-08-16", 189.5),
    (2, "MSFT", "2026-08-16", 401.2),
    (3, "AAPL", "2026-08-17", 191.0),
    (4, "MSFT", "2026-08-17", 405.8),
    (5, "TSLA", "2026-08-17", None),
]
prices = session.createDataFrame(rows, ["id", "ticker", "as_at_date", "close"])
prices.write.mode("overwrite").partitionBy("as_at_date").saveAsTable("demo.prices")
session.table("demo.prices").show()

## 3. The two surfaces are one code path

The DataFrame API and `session.sql()` converge on the same sqlglot tree, so they
optimize, prune and execute identically. Neither is a wrapper around the other.

In [ ]:
api = session.table("demo.prices").filter(F.col("ticker") == "AAPL").select("id", "close")
sql = session.sql("SELECT id, close FROM demo.prices WHERE ticker = 'AAPL'")

print(sorted(r["id"] for r in api.collect()))
print(sorted(r["id"] for r in sql.collect()))

## 4. `explain()` is how you find out what it read

The numbers that matter are ratios. **files** shows partition pruning; **columns**
shows projection pushdown; **pushed filters** shows what Iceberg evaluated against
manifests rather than what DuckDB filtered afterwards.

A filter under *kept in SQL only* still gives the right answer — it just bought no
pruning, which is exactly what someone staring at a slow query needs to be told.

In [ ]:
session.table("demo.prices").filter(F.col("as_at_date") == "2026-08-17").select(
    "ticker", "close"
).explain()

## 5. On a wide table, ask for Arrow

`collect()` builds a `Row` per result row. On the 200-column benchmark table that is
**95% of the wall time** — 25.4 s against 1.3 s for the same query as Arrow
(`FINDINGS.md` §3.8). It is the first thing to reach for on a wide result, and
`toArrowBatches()` also keeps peak memory to one batch.

One rule: **finish iterating before running another query on the session.** DuckDB
ends a result when its cursor runs the next one; icetl notices and refuses rather than
handing you a prefix of the answer.

In [ ]:
table = session.table("demo.prices").toArrow()  # fast path
print(table.num_rows, table.schema.names)

for batch in session.table("demo.prices").toArrowBatches(batchSize=2):
    print("batch of", batch.num_rows)

## 6. `count(*)` opens no file at all

Iceberg records a row count per data file, so an unfiltered count is a sum over
metadata. A filter disqualifies it — file pruning is an over-approximation, so
summing under a predicate would return a number that is too big.

In [ ]:
print("count()          ", session.table("demo.prices").count())  # from manifests
print("count() filtered ", session.table("demo.prices").filter(F.col("close") > 200).count())

## 7. Changing rows in place

`MERGE`, `UPDATE` and `DELETE` are copy-on-write: the files holding matched rows are
rewritten. A table another engine wrote with merge-on-read deletes is **refused**
rather than answered wrongly, because `read_parquet` cannot see a delete file.

In [ ]:
session.sql("UPDATE demo.prices SET close = 999.0 WHERE ticker = 'TSLA'")
session.sql(
    """
    MERGE INTO demo.prices AS t
    USING (SELECT 6 AS id, 'NVDA' AS ticker, '2026-08-17' AS as_at_date, 120.0 AS close) AS s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET close = s.close
    WHEN NOT MATCHED THEN INSERT *
    """
)
session.table("demo.prices").orderBy("id").show()

## 8. Your own Python, as a function

A UDF is registered once and reachable from both surfaces. The return type is
**declared**, not inferred: DuckDB needs it before the first row, and a wrong guess
would be a silently mistyped column.

**NULL does not reach the function by default.** That is a divergence from the
reference, arrived at by measurement rather than preference — `FINDINGS.md` §2.9 has
the matrix. `callOnNull=True` opts into the reference's behaviour for a UDF whose job
is turning NULL into something.

In [ ]:
band = session.udf.register("band", lambda close: "high" if close > 300 else "low", "string")

session.table("demo.prices").select("ticker", band("close").alias("band")).show()
session.sql("SELECT ticker, band(close) AS band FROM demo.prices").show()

In [ ]:
# Vectorised: called once per vector, through Arrow, with a pandas Series.
session.udf.registerVectorised("pct", lambda s: s / 100.0, "double")
session.sql("SELECT ticker, pct(close) AS pct FROM demo.prices").show(3)

# And the opt-in for a function that wants to see NULL itself.
session.udf.register("or_zero", lambda v: 0.0 if v is None else v, "double", callOnNull=True)
session.sql("SELECT or_zero(NULL) AS filled").show()

## 9. Data that is not in a table yet

`session.read.parquet/csv/json` are convenience readers for a file you want to clean up
and append, or join against. The result is an ordinary frame — it just gets no pruning,
because there are no manifests to prune with.

**Watch the inferred types.** DuckDB's CSV sniffer is good, which is the problem: it
reads `2026-08-17` as a `date` while the table stores a `string`, and an append with a
mismatched schema is refused. Casting to the table's schema is the fix, and it is worth
doing deliberately rather than discovering later.

In [ ]:
csv_path = root / "extra.csv"
csv_path.write_text("id,ticker,as_at_date,close\n7,AMD,2026-08-17,142.5\n", encoding="utf-8")

extra = session.read.csv(str(csv_path).replace("\\", "/"), header=True)
print("as sniffed:", extra.schema.simpleString())
print("as stored :", session.table("demo.prices").schema.simpleString())

aligned = extra.select(
    "id", "ticker", F.col("as_at_date").cast("string").alias("as_at_date"), "close"
)
aligned.write.mode("append").saveAsTable("demo.prices")
print("rows now:", session.table("demo.prices").count())

## 10. Time travel

Every write is a snapshot, and a snapshot is readable. `VERSION AS OF` takes an id,
`TIMESTAMP AS OF` takes a moment, and `session.read.option(...)` is the same thing
through the reader.

Iceberg's metadata tables are queryable too: `demo.prices.snapshots`,
`.files`, `.manifests`, `.history`, `.partitions`.

In [ ]:
snapshots = session.sql(
    "SELECT snapshot_id, operation FROM demo.prices.snapshots ORDER BY committed_at"
).collect()
for row in snapshots:
    print(row["snapshot_id"], row["operation"])

first = snapshots[0]["snapshot_id"]
print(
    "rows at the first snapshot:",
    session.sql(f"SELECT count(*) AS n FROM demo.prices VERSION AS OF {first}").collect()[0]["n"],
)

## 11. Keeping the table fast

Compaction is not tidying. The read design leans on **column-statistics file pruning**,
and statistics only prune when files are large and sorted — so many small files defeat
it. The usual order is compact, expire, then remove orphans.

`removeOrphanFiles` **reports by default**: deleting a file that only looks orphaned is
unrecoverable, and a commit in flight has written its data before referencing it.

In [ ]:
maintenance = session.maintenance("demo.prices")

print(maintenance.compact(targetFileSizeBytes=10_000_000))
print(maintenance.expireSnapshots(retainLast=2))
print(maintenance.removeOrphanFiles())  # dry run; pass dryRun=False to delete

## 12. Where to look next

| | |
|---|---|
| [`README.md`](../README.md) | setup, the test gate, the benchmark harness |
| [`PLAN.md`](../PLAN.md) | the design, the phases, and the decisions behind them |
| [`STATUS.md`](../STATUS.md) | what is built, phase by phase |
| [`FINDINGS.md`](../FINDINGS.md) | every dependency trap and wrong-answer bug found so far |
| [`BENCHMARKS.md`](../BENCHMARKS.md) | numbers, and how to read a regression |
| [`compat/divergence.md`](../src/icetl/compat/divergence.md) | every place icetl and the reference differ |

**Ten minutes in `FINDINGS.md` is worth it before writing anything that generates SQL.**
It is a register of the times a dependency was confidently wrong.

In [ ]:
session.stop()